# Part D: Policy RAG Co-Pilot

**Goal:** Answer customer questions from company policies.

**Key techniques:** Retrieval-Augmented Generation (RAG), LLM, escalation to humans (plus a refund agent).

## How to use this notebook
1. Keep this notebook in the **project root** (next to `generate_data.py`, `src/`, `data/`). It also works from a sub-folder; the setup cell finds the root.
2. Select the project's `.venv` as the kernel (top-right of the notebook in VS Code).
3. Run the cells **top to bottom** (`Run All`). Later cells depend on earlier ones.

## What is inside
The code in each section is the code from your `.py` file, split into cells with no change to the logic. Sections run in this order:

1. `policy_rag.py` - split policies into chunks and build a TF-IDF index
2. `policy_assistant.py` - answer questions from retrieved chunks and evaluate
3. `compare_policy_overlap.py` - effect of chunk overlap
4. `refund_agent.py` - tool-using refund agent with checkpoints and safety rules

**Note:** the refund agent saves a checkpoint and ledger in `state/` and `outputs/`, exactly like the script does.

The original `.py` files are untouched and are still needed: some sections import each other (for example `from policy_rag import ...`) and `mock_llm.py` is the supplied local LLM.


In [ ]:
# Setup cell: lets the original script code run unchanged inside a notebook.
# 1) finds the project root, 2) makes it the working directory (the scripts use
# relative paths like "data/raw"), 3) makes the project's own modules importable,
# 4) defines __file__ (scripts use it, notebooks do not have it).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "mock_llm.py").exists():
        ROOT = candidate
        break

os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

__file__ = str(ROOT / "src" / "notebook.py")
print("Project root:", ROOT)

---
## `src/policy_rag.py`
**Concept:** RAG = first *retrieve* the most relevant policy text, then let the LLM answer only from it. **How:** policies are cut into overlapping chunks, turned into TF-IDF vectors, and matched to a question by cosine similarity.

In [ ]:
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
PROJECT_ROOT = Path(__file__).resolve().parent.parent
POLICY_DIR = PROJECT_ROOT / "data" / "policies"
CHUNK_SIZE = 40
CHUNK_OVERLAP = 10

In [ ]:
def load_policy_chunks(overlap=CHUNK_OVERLAP):
    """Load policy files and split them into configurable word chunks.

    Args:
        overlap: Number of words shared between consecutive chunks.
                 Defaults to 10; use 0 for the zero-overlap evaluation.
    """
    if not isinstance(overlap, int) or not 0 <= overlap < CHUNK_SIZE:
        raise ValueError(
            f"overlap must be an integer from 0 to {CHUNK_SIZE - 1}"
        )

    if not POLICY_DIR.exists():
        raise FileNotFoundError(
            f"Policy directory not found: {POLICY_DIR}"
        )

    chunks = []
    step = CHUNK_SIZE - overlap

    for path in sorted(POLICY_DIR.glob("*.txt")):
        text = path.read_text(encoding="utf-8")
        words = text.split()

        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]

            if not chunk_words:
                break

            chunks.append({
                "source": path.name,
                "chunk_id": len(chunks) + 1,
                "text": " ".join(chunk_words),
            })

            # Stop after the first chunk that reaches the document end.
            if start + CHUNK_SIZE >= len(words):
                break

    return chunks

In [ ]:
def build_index(chunks):
    """Create a TF-IDF index from the policy chunks."""
    if not chunks:
        raise ValueError("No policy chunks found.")

    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(
        [chunk["text"] for chunk in chunks]
    )

    return vectorizer, matrix

In [ ]:
def retrieve(query, chunks, vectorizer, matrix, top_k=3):
    """Return the most relevant policy chunks using cosine similarity."""
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, matrix).flatten()

    ranked_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            **chunks[index],
            "score": round(float(scores[index]), 4),
        })

    return results

In [ ]:
def main():
    # The normal project configuration remains 40 words / 10 overlap.
    chunks = load_policy_chunks()
    vectorizer, matrix = build_index(chunks)

    print("Policy files:", len(list(POLICY_DIR.glob("*.txt"))))
    print("Total chunks:", len(chunks))
    print("Chunk size:", CHUNK_SIZE, "words")
    print("Chunk overlap:", CHUNK_OVERLAP, "words")

    query = "refund policy late delivery"
    results = retrieve(query, chunks, vectorizer, matrix, top_k=3)

    print(f"\nQuery: {query}")
    print("\nTop 3 retrieved chunks:")

    for rank, result in enumerate(results, start=1):
        print(f"\n{rank}. {result['source']}")
        print("Chunk ID:", result["chunk_id"])
        print("Similarity:", result["score"])
        print("Text:", result["text"])

    print("\nSource file counts:")
    for source in sorted({chunk["source"] for chunk in chunks}):
        count = sum(chunk["source"] == source for chunk in chunks)
        print(f"{source}: {count} chunks")

    refund_results = [
        result for result in results
        if result["source"] == "refund_policy.txt"
    ]

    if CHUNK_OVERLAP == 10:
        if len(chunks) != 15:
            raise AssertionError(
                f"Expected 15 chunks, got {len(chunks)}"
            )

        if len(refund_results) != 3:
            raise AssertionError(
                "Expected all top 3 results to come from refund_policy.txt"
            )

        print("\nPASS: 15 chunks created.")
        print("PASS: All top 3 results came from refund_policy.txt.")

In [ ]:
if __name__ == "__main__":
    main()

---
## `src/policy_assistant.py`
**How:** retrieves chunks, asks the mock LLM to answer, cleans the answer, escalates to a human when the policy has no answer, and scores results against `policy_eval.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path(__file__).resolve().parent.parent
SRC_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(SRC_DIR))
import mock_llm
from policy_rag import load_policy_chunks, build_index, retrieve
EVALUATION_FILE = (
    PROJECT_ROOT / "data" / "raw" / "policy_eval.csv"
)
OUTPUT_FILE = (
    PROJECT_ROOT / "outputs" / "policy_eval_results.csv"
)

In [ ]:
def clean_answer(answer):
    """Remove repeated phrases while preserving important policy facts."""
    import re

    answer = re.sub(r"\s+", " ", str(answer)).strip()
    words = answer.split()

    # Remove a repeated phrase only when the full phrase appears twice.
    # Start with longer phrases to avoid removing individual key words.
    for size in range(min(18, len(words) // 2), 3, -1):
        changed = True

        while changed:
            changed = False
            n = len(words)

            for start in range(n - size + 1):
                phrase = [
                    word.casefold()
                    for word in words[start:start + size]
                ]

                for later in range(start + size, n - size + 1):
                    candidate = [
                        word.casefold()
                        for word in words[later:later + size]
                    ]

                    if phrase == candidate:
                        del words[later:later + size]
                        changed = True
                        break

                if changed:
                    break

    return " ".join(words).strip()

In [ ]:
def answer_question(question, chunks, vectorizer, matrix):
    """Retrieve relevant policy chunks and ask the offline mock LLM."""

    results = retrieve(
        query=question,
        chunks=chunks,
        vectorizer=vectorizer,
        matrix=matrix,
        top_k=3,
    )

    context_parts = []

    for result in results:
        context_parts.append(
            f"[{result['source']}]\n{result['text']}"
        )

    context = "\n\n".join(context_parts)

    # Keep exactly two messages: system and user.
    messages = [
        {
            "role": "system",
            "content": (
                "POLICY ASSISTANT\n"
                "Answer the user's question using only the supplied "
                "policy context. Do not invent policy details."
            ),
        },
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{context}\n"
                f"QUESTION:\n{question}"
            ),
        },
    ]

    response = mock_llm.chat(messages)

    original_answer = response.choices[0].message.content
    answer = clean_answer(original_answer)

    return answer, results, response.usage.total_tokens

In [ ]:
def main():
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

    evaluation = pd.read_csv(
        EVALUATION_FILE,
        keep_default_na=False,
    )

    chunks = load_policy_chunks()
    vectorizer, matrix = build_index(chunks)

    evaluation_results = []

    # Reset mock LLM state before the evaluation run.
    mock_llm.reset()

    for _, row in evaluation.iterrows():
        question = str(row["question"])
        expected_doc = str(row["expected_doc"])
        expected_fact = str(row["expected_fact"])

        answer, retrieved, tokens_used = answer_question(
            question,
            chunks,
            vectorizer,
            matrix,
        )

        retrieved_docs = sorted({
            item["source"] for item in retrieved
        })

        is_answerable = expected_doc != "NONE"

        document_hit = (
            expected_doc in retrieved_docs
            if is_answerable
            else None
        )

        # Keep the required case-sensitive expected-fact check.
        fact_found = (
            expected_fact in answer
            if is_answerable and expected_fact
            else None
        )

        fallback_correct = (
            answer.startswith("I could not find")
            if not is_answerable
            else None
        )

        evaluation_results.append({
            "question": question,
            "expected_doc": expected_doc,
            "expected_fact": expected_fact,
            "retrieved_docs": ";".join(retrieved_docs),
            "document_hit": document_hit,
            "answer": answer,
            "fact_found_in_answer": fact_found,
            "fallback_correct": fallback_correct,
            "tokens_used": tokens_used,
        })

        print("\nQuestion:", question)
        print("Expected document:", expected_doc)
        print("Retrieved documents:", ", ".join(retrieved_docs))
        print("Answer:", answer)
        print("Expected fact found in answer:", fact_found)

        if not is_answerable:
            print("Correct fallback:", fallback_correct)

    results_df = pd.DataFrame(evaluation_results)
    results_df.to_csv(OUTPUT_FILE, index=False)

    answerable = results_df[
        results_df["expected_doc"] != "NONE"
    ]

    out_of_scope = results_df[
        results_df["expected_doc"] == "NONE"
    ]

    print("\n" + "=" * 60)
    print("Policy evaluation complete")
    print("Questions evaluated:", len(results_df))

    print(
        "Expected document retrieved:",
        f"{int(answerable['document_hit'].sum())}/{len(answerable)}",
    )

    print(
        "Expected fact found (case-sensitive):",
        f"{int(answerable['fact_found_in_answer'].sum())}"
        f"/{len(answerable)}",
    )

    print(
        "Correct out-of-scope fallbacks:",
        f"{int(out_of_scope['fallback_correct'].sum())}"
        f"/{len(out_of_scope)}",
    )

    print("Saved:", OUTPUT_FILE)

In [ ]:
if __name__ == "__main__":
    main()

---
## `src/compare_policy_overlap.py`
**Why:** shows whether overlapping chunks help retrieval by re-running the evaluation with `overlap=0`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from policy_rag import load_policy_chunks, build_index
from policy_assistant import answer_question
EVALUATION_FILE = PROJECT_ROOT / "data" / "raw" / "policy_eval.csv"
OUTPUT_FILE = PROJECT_ROOT / "outputs" / "policy_eval_zero_overlap_results.csv"

In [ ]:
def main():
    evaluation = pd.read_csv(EVALUATION_FILE, keep_default_na=False)

    # Build the index without overlapping words.
    chunks = load_policy_chunks(overlap=0)
    vectorizer, matrix = build_index(chunks)

    print("Chunk overlap: 0 words")
    print("Total chunks:", len(chunks))

    results = []

    for _, row in evaluation.iterrows():
        question = row["question"]
        expected_doc = row["expected_doc"]
        expected_fact = row["expected_fact"]

        answer, retrieved, tokens_used = answer_question(
            question,
            chunks,
            vectorizer,
            matrix,
        )

        retrieved_docs = sorted({
            item["source"] for item in retrieved
        })

        is_answerable = expected_doc != "NONE"

        document_hit = (
            expected_doc in retrieved_docs
            if is_answerable else None
        )

        # Check the expected fact case-sensitively.
        fact_found = (
            expected_fact in answer
            if is_answerable and expected_fact else None
        )

        fallback_correct = (
            answer.startswith("I could not find")
            if not is_answerable else None
        )

        results.append({
            "question": question,
            "expected_doc": expected_doc,
            "retrieved_docs": ";".join(retrieved_docs),
            "document_hit": document_hit,
            "expected_fact": expected_fact,
            "fact_found_in_answer": fact_found,
            "fallback_correct": fallback_correct,
            "answer": answer,
            "tokens_used": tokens_used,
        })

    results_df = pd.DataFrame(results)
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUTPUT_FILE, index=False)

    answerable = results_df[
        results_df["expected_doc"] != "NONE"
    ]
    out_of_scope = results_df[
        results_df["expected_doc"] == "NONE"
    ]

    print("\n=== ZERO-OVERLAP EVALUATION ===")
    print("Total questions:", len(results_df))
    print(
        "Expected document retrieved:",
        f"{int(answerable['document_hit'].sum())}/{len(answerable)}",
    )
    print(
        "Expected fact found (case-sensitive):",
        f"{int(answerable['fact_found_in_answer'].sum())}/{len(answerable)}",
    )
    print(
        "Correct out-of-scope fallbacks:",
        f"{int(out_of_scope['fallback_correct'].sum())}/{len(out_of_scope)}",
    )

    failed = answerable[
        (answerable["document_hit"] != True)
        | (answerable["fact_found_in_answer"] != True)
    ]

    print("\nAnswerable questions that failed:")
    if failed.empty:
        print("None. Inspect the results and compare with overlapping chunks.")
    else:
        for _, row in failed.iterrows():
            print("\nQuestion:", row["question"])
            print("Expected fact:", row["expected_fact"])
            print("Retrieved document:", row["retrieved_docs"])
            print("Answer:", row["answer"])

    print("\nSaved results:", OUTPUT_FILE)

In [ ]:
if __name__ == "__main__":
    main()

---
## `src/refund_agent.py`
**How:** an agent loop that uses tools (order lookup, policy retrieval, refund) with iteration and token limits, a crash-safe checkpoint, and rules that send risky cases to a human.

In [ ]:
import json
import re
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
import mock_llm
from policy_rag import load_policy_chunks, build_index, retrieve
DB_PATH = PROJECT_ROOT / "data" / "instamart.db"
SCENARIOS_PATH = PROJECT_ROOT / "data" / "raw" / "agent_scenarios.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
STATE_DIR = PROJECT_ROOT / "state"
RUNS_PATH = OUTPUT_DIR / "agent_runs.jsonl"
LEDGER_PATH = OUTPUT_DIR / "refund_ledger.jsonl"
CHECKPOINT_PATH = STATE_DIR / "agent_checkpoint.json"
MAX_ITERATIONS = 6
TOKEN_LIMIT = 6000

In [ ]:
def read_json(path, default):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(data, indent=2, default=str),
        encoding="utf-8",
    )
    temporary.replace(path)

In [ ]:
def append_jsonl_once(path, record, unique_key):
    """Append a JSON record only when its unique key is not present."""
    path.parent.mkdir(parents=True, exist_ok=True)

    existing = []
    if path.exists():
        with path.open(encoding="utf-8") as file:
            for line in file:
                if line.strip():
                    existing.append(json.loads(line))

    if any(item.get(unique_key) == record.get(unique_key)
           for item in existing):
        return

    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, default=str) + "\n")

In [ ]:
def get_order(order_id):
    """Read order information from SQLite."""
    import sqlite3

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        row = connection.execute(
            """
            SELECT order_id, customer_id, status,
                   order_amount, delivery_minutes
            FROM orders
            WHERE order_id = ?
            LIMIT 1
            """,
            (order_id,),
        ).fetchone()

    if row is None:
        return {"error": "Order not found", "order_id": order_id}

    return dict(row)

In [ ]:
def search_policy(query):
    """Retrieve relevant chunks from the local policy documents."""
    chunks = load_policy_chunks()
    vectorizer, matrix = build_index(chunks)
    matches = retrieve(query, chunks, vectorizer, matrix, top_k=3)

    return {
        "results": [
            {"source": item["source"], "text": item["text"]}
            for item in matches
        ]
    }

In [ ]:
def calculate_refund(order_amount, delivery_minutes, status):
    """Apply completed-order and late-delivery eligibility rules."""
    if str(status).lower() != "completed":
        amount = 0.0
        reason = "Only completed orders are eligible."
    elif delivery_minutes is None or pd.isna(delivery_minutes):
        amount = 0.0
        reason = "Delivery time is unavailable."
    elif float(delivery_minutes) <= 45:
        amount = 0.0
        reason = "Delivery was not more than 45 minutes."
    else:
        amount = round(float(order_amount), 2)
        reason = "Completed order delivered more than 45 minutes."

    return {
        "refund_inr": amount,
        "reason": reason,
    }

In [ ]:
def issue_refund(order_id, refund_inr, scenario_id):
    """Record an automatic refund with an idempotency key."""
    record = {
        "scenario_id": scenario_id,
        "order_id": order_id,
        "amount_inr": round(float(refund_inr), 2),
        "status": "issued",
    }
    append_jsonl_once(LEDGER_PATH, record, "scenario_id")
    return {"refund_inr": record["amount_inr"], "status": "issued"}

In [ ]:
def escalate_to_human(order_id, refund_inr, scenario_id):
    """Record a refund requiring human approval."""
    return {
        "scenario_id": scenario_id,
        "order_id": order_id,
        "refund_inr": round(float(refund_inr), 2),
        "status": "escalated",
        "reason": "Refund above INR 1000 requires human approval.",
    }

In [ ]:
def execute_tool(name, arguments, scenario_id, order):
    """Dispatch a mock LLM tool call to a local Python function."""
    if name == "get_order":
        return get_order(arguments["order_id"])

    if name == "search_policy":
        return search_policy(arguments["query"])

    if name == "calculate_refund":
        if order is None or "error" in order:
            return {"refund_inr": 0.0, "reason": "Order unavailable."}

        return calculate_refund(
            order_amount=arguments["order_amount"],
            delivery_minutes=arguments["delivery_minutes"],
            status=order["status"],
        )

    if name == "issue_refund":
        return issue_refund(
            arguments["order_id"],
            arguments["refund_inr"],
            scenario_id,
        )

    if name == "escalate_to_human":
        return escalate_to_human(
            arguments["order_id"],
            arguments["refund_inr"],
            scenario_id,
        )

    return {"error": f"Unknown tool: {name}"}

In [ ]:
def get_order_from_messages(messages):
    for message in messages:
        if message.get("role") == "tool" and message.get("name") == "get_order":
            return json.loads(message["content"])
    return None

In [ ]:
def get_terminal_status(answer):
    text = (answer or "").upper()

    if text.startswith("REFUND:"):
        return "REFUND"
    if text.startswith("ESCALATE:"):
        return "ESCALATE"
    if text.startswith("DENIED:"):
        return "DENIED"
    if text.startswith("NOT_FOUND:"):
        return "NOT_FOUND"
    if text.startswith("NEEDS_INFO:"):
        return "NEEDS_INFO"

    return "FALLBACK_INVALID_RESPONSE"

In [ ]:
def save_checkpoint(checkpoint):
    write_json(CHECKPOINT_PATH, checkpoint)

In [ ]:
def run_scenario(
    scenario,
    checkpoint,
    *,
    token_limit=TOKEN_LIMIT,
    simulate_crash=False,
    simulate_timeout=False,
):
    """Run or resume one scenario, saving progress after every tool."""
    scenario_id = str(scenario["scenario_id"])
    state = checkpoint["scenarios"].get(scenario_id)

    if state and state.get("completed"):
        return state["result"]

    if state is None:
        order_match = re.search(r"ORD-\d{6}", str(scenario["message"]))
        user_message = str(scenario["message"])

        # S7 deliberately tests the maximum-iteration fallback.
        if scenario_id == "S7" and "[LOOP]" not in user_message:
            user_message += " [LOOP]"

        state = {
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "AGENT\n"
                        "Use tools to check the order, consult the policy, "
                        "calculate eligibility, then refund or escalate."
                    ),
                },
                {"role": "user", "content": user_message},
            ],
            "tokens_used": 0,
            "tool_names": [],
            "iterations": 0,
            "completed": False,
            "order_id": order_match.group(0) if order_match else None,
        }
        checkpoint["scenarios"][scenario_id] = state
        save_checkpoint(checkpoint)

    if simulate_timeout:
        result = {
            "scenario_id": scenario_id,
            "status": "FALLBACK_TIMEOUT",
            "answer": "Human review required after simulated timeout.",
            "tokens_used": state["tokens_used"],
            "tool_names": state["tool_names"],
        }
        state["completed"] = True
        state["result"] = result
        save_checkpoint(checkpoint)
        return result

    while state["iterations"] < MAX_ITERATIONS:
        if state["tokens_used"] >= token_limit:
            result = {
                "scenario_id": scenario_id,
                "status": "FALLBACK_TOKEN_LIMIT",
                "answer": "Human review required: token limit reached.",
                "tokens_used": state["tokens_used"],
                "tool_names": state["tool_names"],
            }
            state["completed"] = True
            state["result"] = result
            save_checkpoint(checkpoint)
            return result

        response = mock_llm.chat(state["messages"])
        state["tokens_used"] += response.usage.total_tokens
        state["iterations"] += 1

        message = response.choices[0].message
        tool_calls = message.tool_calls or []

        if not tool_calls:
            answer = message.content or ""
            result = {
                "scenario_id": scenario_id,
                "status": get_terminal_status(answer),
                "answer": answer,
                "tokens_used": state["tokens_used"],
                "tool_names": state["tool_names"],
            }
            state["completed"] = True
            state["result"] = result
            save_checkpoint(checkpoint)
            return result

        # Save the assistant's tool request before executing the tool.
        state["messages"].append({
            "role": "assistant",
            "content": message.content,
            "tool_calls": [
                {
                    "id": call.id,
                    "name": call.name,
                    "arguments": call.arguments,
                }
                for call in tool_calls
            ],
        })
        save_checkpoint(checkpoint)

        for call in tool_calls:
            # Resume safety: a completed tool call ID is already represented
            # by its tool message. Never execute it a second time.
            completed_ids = {
                item.get("tool_call_id")
                for item in state["messages"]
                if item.get("role") == "tool"
            }
            if call.id in completed_ids:
                continue

            order = get_order_from_messages(state["messages"])
            arguments = call.arguments

            tool_result = execute_tool(
                call.name,
                arguments,
                scenario_id,
                order,
            )

            state["tool_names"].append(call.name)
            state["messages"].append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": call.name,
                "content": json.dumps(tool_result, default=str),
            })
            save_checkpoint(checkpoint)

            # S1 simulates a crash after its first two tool executions.
            # The checkpoint is left intact for the next run to resume.
            if (
                simulate_crash
                and scenario_id == "S1"
                and len(state["tool_names"]) == 2
            ):
                print("Simulated crash after two tools for S1.")
                return {
                    "scenario_id": scenario_id,
                    "status": "INTERRUPTED_SIMULATED_CRASH",
                    "answer": "Checkpoint saved; rerun to resume.",
                    "tokens_used": state["tokens_used"],
                    "tool_names": state["tool_names"],
                }

    result = {
        "scenario_id": scenario_id,
        "status": "FALLBACK_MAX_ITERATIONS",
        "answer": "Human review required: maximum iterations reached.",
        "tokens_used": state["tokens_used"],
        "tool_names": state["tool_names"],
    }
    state["completed"] = True
    state["result"] = result
    save_checkpoint(checkpoint)
    return result

In [ ]:
def write_run_log(checkpoint):
    """Rebuild the run log from completed checkpoint entries."""
    records = [
        state["result"]
        for state in checkpoint["scenarios"].values()
        if state.get("completed") and state.get("result")
    ]

    with RUNS_PATH.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, default=str) + "\n")

In [ ]:
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    STATE_DIR.mkdir(parents=True, exist_ok=True)

    scenarios_df = pd.read_csv(SCENARIOS_PATH)
    checkpoint = read_json(CHECKPOINT_PATH, {"scenarios": {}})

    # First, demonstrate a crash after two tools on S1.
    s1_rows = scenarios_df[scenarios_df["scenario_id"] == "S1"]
    if not s1_rows.empty:
        print("\n--- S1 crash/resume demonstration ---")
        interrupted = run_scenario(
            s1_rows.iloc[0].to_dict(),
            checkpoint,
            simulate_crash=True,
        )
        print("First attempt:", interrupted["status"])

        resumed = run_scenario(
            s1_rows.iloc[0].to_dict(),
            checkpoint,
        )
        print("Resumed result:", resumed["status"])
        print("S1 tools:", resumed["tool_names"])
        print("S1 tool execution count:", len(resumed["tool_names"]))

    # Run the remaining scenarios, resuming saved work if needed.
    print("\n--- Running agent scenarios ---")
    for _, row in scenarios_df.iterrows():
        scenario = row.to_dict()
        scenario_id = str(scenario["scenario_id"])

        if scenario_id == "S1":
            continue

        result = run_scenario(scenario, checkpoint)
        print(scenario_id, "->", result["status"])

    # Separate deterministic demonstrations for the two other fallbacks.
    if not scenarios_df.empty:
        demo = scenarios_df.iloc[0].to_dict()

        token_result = run_scenario(
            {**demo, "scenario_id": "DEMO_TOKEN_LIMIT"},
            checkpoint,
            token_limit=1,
        )
        print("Token demo:", token_result["status"])

        timeout_result = run_scenario(
            {**demo, "scenario_id": "DEMO_TIMEOUT"},
            checkpoint,
            simulate_timeout=True,
        )
        print("Timeout demo:", timeout_result["status"])

    write_run_log(checkpoint)

    print("\nRun log:", RUNS_PATH)
    print("Refund ledger:", LEDGER_PATH)
    print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
if __name__ == "__main__":
    main()